# 06 — Query Workload dan Ground Truth (Sementara)

**Tahap 5–6 ceklis: Query workload dan ground truth** · Protokol bagian 14, 15 (K-04), 18, 19; README bagian 15–17

**Status: SEMENTARA.** Dibangun dengan pilihan yang menunggu bimbingan (lihat `configs/workload.yaml`):
- batas bin usulan protokol: B1 < 0,1% · B2 0,1–0,5% · B3 0,5–2,5% · B4 2,5–10% · B5 > 10%;
- **set utama**: predikat dengan ≥ 10 × k = 100 titik (protokol bagian 18);
- **set ekstensi**: predikat dengan 10–99 titik (≥ k), disiapkan bila syarat kecukupan dilonggarkan;
- berita yang tertaut ke beberapa wilayah/kategori cocok bila salah satu nilainya cocok (`any_match`);
- kolom provinsi hasil perapian (menunggu K-01).

**Kueri:** dokumen korpus yang ditahan (≥ 4 kata, bukan 100 kueri development), separuh memenuhi predikat dan separuh tidak. **Eksklusi self-match:** dokumen kueri, dokumen bervektor identik, dan dokumen satu `story_lineage_id` dikeluarkan dari himpunan eligible.

**Dibangun dengan:** `python -m src.workload.build_queries` lalu `python -m src.groundtruth.build_groundtruth`.

## Langkah 1: Membaca populasi predikat, workload, dan ground truth

In [ ]:
# Populasi predikat (audit), pasangan kueri-filter beku, dan ground truth eksak
import json
import numpy as np
import pandas as pd

populasi = pd.read_parquet("../data/processed/filters.parquet")
workload = pd.read_parquet("../data/processed/workload.parquet")
gt = pd.read_parquet("../data/groundtruth/filtered_knn_k10.parquet")
korpus = pd.read_parquet("../data/processed/corpus.parquet", columns=["doc_id", "source_type", "text"])

print("Populasi predikat:", len(populasi))
print("Layak set utama (>= 100 titik):", populasi["layak_utama"].sum(), "| layak set ekstensi (10-99 titik):", populasi["layak_ekstensi"].sum())
print("Pasangan kueri-filter:", len(workload), "| kueri unik:", workload["query_row"].nunique(), "| ground truth:", len(gt))

## Langkah 2: Populasi predikat layak per tipe dan bin

Jumlah predikat yang memenuhi syarat kecukupan, sebelum sampling. B1 pada set utama selalu kosong karena 100 titik = 0,155% > batas atas B1 (0,1%).

In [ ]:
# Jumlah predikat layak per tipe dan bin, untuk set utama dan set ekstensi
layak_utama = pd.crosstab(populasi.loc[populasi["layak_utama"], "type"], populasi.loc[populasi["layak_utama"], "bin"])
layak_ekstensi = pd.crosstab(populasi.loc[populasi["layak_ekstensi"], "type"], populasi.loc[populasi["layak_ekstensi"], "bin"])
print("Set utama:")
display(layak_utama)
print("Set ekstensi:")
display(layak_ekstensi)

## Langkah 3: Komposisi workload per set, bin, dan tipe predikat

Protokol bagian 18 mensyaratkan setiap bin berisi kueri yang memadai (`min_queries_per_bin` = 30) dan, bila memungkinkan, lebih dari satu tipe predikat.

In [ ]:
# Jumlah pasangan per tipe predikat untuk setiap (set, bin)
komposisi = workload.pivot_table(index="type", columns=["set", "bin"], values="pair_id", aggfunc="count", fill_value=0)
display(komposisi)

# Jumlah tipe predikat dan pasangan per (set, bin)
per_bin = workload.groupby(["set", "bin"]).agg(pasangan=("pair_id", "count"), tipe=("type", "nunique"), predikat=("filter_id", "nunique"))
per_bin

## Langkah 4: Median dan rentang match rate aktual per bin

Pelaporan wajib menurut protokol bagian 14 butir 5 dan bagian 15 (K-04).

In [ ]:
# Match rate (%) per pasangan: median, minimum, maksimum, dan jumlah titik eligible setelah eksklusi
mr = workload.assign(match_rate_pct=workload["match_rate"] * 100)
ringkas_mr = mr.groupby(["set", "bin"]).agg(
    median_pct=("match_rate_pct", "median"),
    min_pct=("match_rate_pct", "min"),
    maks_pct=("match_rate_pct", "max"),
    eligible_min=("n_eligible", "min"),
    eligible_median=("n_eligible", "median"),
).round(4)
ringkas_mr

## Langkah 5: Pemeriksaan kueri dan ground truth

- Keseimbangan kueri yang memenuhi / tidak memenuhi predikat (analisis korelasi kueri–filter).
- Tidak ada titik eksklusi self-match di ground truth.
- Jarak ground truth: kueri yang memenuhi predikat seharusnya punya tetangga eligible lebih dekat.

In [ ]:
# Proporsi kueri yang memenuhi predikat per (set, bin)
display(workload.pivot_table(index=["set", "bin"], columns="query_in_filter", values="pair_id", aggfunc="count"))

# Kebocoran eksklusi ke ground truth (harus 0)
gabung = workload.merge(gt, on="pair_id")
bocor = sum(len(set(r.gt_rows) & set(r.excluded_rows)) for r in gabung.itertuples())
print("Titik eksklusi yang muncul di ground truth:", bocor)

# Median jarak ground truth ke-1 dan ke-10, menurut apakah kueri memenuhi predikat
gabung["jarak_1"] = gabung["gt_dist"].str[0]
gabung["jarak_10"] = gabung["gt_dist"].str[-1]
gabung.groupby(["set", "bin", "query_in_filter"])[["jarak_1", "jarak_10"]].median().round(3)

## Ringkasan hasil (dari run uji 14 Okt 2026; periksa ulang setelah dijalankan di laptop)

| Set | Bin | Pasangan | Tipe predikat |
|---|---|---:|---:|
| Utama (≥ 100 titik) | B2 | 300 | 5 |
| Utama | B3 | 360 | 6 |
| Utama | B4 | 420 | 7 |
| Utama | B5 | 360 | 6 |
| Ekstensi (10–99 titik) | B1 | 291 | 5 |
| Ekstensi | B2 (0,1–0,155%) | 303 | 5 |

- **2.034 pasangan** dari 1.987 kueri unik; 5.448 kandidat predikat, 928 layak set utama, 1.507 layak set ekstensi.
- Setiap sel (set, bin, tipe) berisi 54–63 pasangan, memenuhi syarat ≥ 30 kueri per bin.
- Setiap bin berisi ≥ 5 tipe predikat, sesuai protokol bagian 18.
- Tidak ada titik eksklusi self-match yang bocor ke ground truth; setiap pasangan memiliki tepat 10 titik ground truth.

## Implikasi (draf)

1. **Set utama memenuhi seluruh syarat protokol bagian 18**, tetapi tidak mencakup rezim < 0,155%.
2. **Set ekstensi mencakup rezim 0,0155–0,155%**, termasuk rentang 0,01–0,1% tempat paper melaporkan TwoHop runtuh. Set ini hanya dipakai bila syarat kecukupan dilonggarkan saat bimbingan; bila tidak, set ini dilaporkan terpisah sebagai analisis tambahan atau tidak dipakai.
3. **Pembekuan resmi** (batas bin, himpunan kueri) dilakukan setelah keputusan bimbingan; sebelum itu workload ini tidak boleh dipakai untuk hasil utama.